<a href="https://colab.research.google.com/github/deruke/aisecops-labs/blob/main/notebooks/Lab9_Security_Agent_Tool_Calling.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9: Building a Security Agent with Tool Calling
## Teaching an LLM to Use Security Tools Autonomously

**Workshop:** Attacking, Defending & Leveraging Agentic AI
**Authors:** Derek Banks and Dr. Brian Fehrman
**Time:** 60 minutes
**Platform:** Google Colab (T4 recommended)
**Target Model:** Qwen2.5-3B-Instruct (4-bit quantized)

---

### What You Will Learn
- How to build an AI agent that uses **tool calling** to perform security tasks
- The iterative agentic loop: **Plan -> Act -> Observe -> Reflect**
- How the agent **reasons** about which tool to call and **chains** results together
- The real risks of **excessive agency** in security automation

### Prerequisites
- Basic understanding of LLMs and prompt engineering
- Familiarity with Python
- Completion of Labs 1-8 (or equivalent background)
- A Google account for Colab access

---
## Cell 1: Environment Setup
---

In [ ]:
# Cell 1: Environment Setup
!pip install -q transformers accelerate bitsandbytes torch requests

import torch, sys, json, re, time, random, hashlib
from datetime import datetime, timedelta

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime -> Change runtime type -> T4 GPU")
print("Setup complete.")


In [ ]:
# Cell 2: Load the Language Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

try:
    print(f"Loading {MODEL_ID} in 4-bit...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True,
    )
    print(f"Loaded: {MODEL_ID}")
except Exception as e:
    MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    print(f"Falling back to {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True,
    )

param_count = sum(p.numel() for p in model.parameters())
print(f"Parameters: {param_count / 1e9:.2f}B -- Agent brain ready.")


---
## Background: What Is an AI Agent?
---

An **AI agent** is an LLM that can take **actions** in the world, not just generate text.

| | Chatbot | Agent |
|---|---------|-------|
| Input | User message | User message + tool results |
| Output | Text reply | Text reply OR tool call |
| State | Stateless or simple history | Maintains investigation state |
| Loop | Single turn | Multi-turn with tool execution |

### The Agentic Loop

```
                    +------------------+
                    |   User Query     |
                    +--------+---------+
                             |
                             v
                    +------------------+
               +--->|    OBSERVE       |  <-- Receive input or tool results
               |    +--------+---------+
               |             |
               |             v
               |    +------------------+
               |    |     THINK        |  <-- Reason about what to do next
               |    +--------+---------+
               |             |
               |             v
               |    +------------------+
               |    |      ACT         |  <-- Call a tool OR give final answer
               |    +--------+---------+
               |             |
               |       Tool call?
               |      /          \
               |    YES           NO
               |    /               \
               |   v                 v
               |  Execute         +------------------+
               |  Tool            |  FINAL ANSWER    |
               |   |              +------------------+
               +---+
```

### How Tool Calling Works

1. We define **tools** -- Python functions that do specific things (look up IPs, search CVEs, etc.)
2. We tell the LLM about these tools in its **system prompt**
3. When the LLM needs information, it outputs a **structured tool call** instead of text
4. Our code **parses** the call, **executes** it, and feeds the result back
5. The LLM **continues reasoning** with the new information

---
## Part 1: Building Security Tools
---

In [ ]:
# Cell 3: Security Tools -- ip_reputation, cve_lookup, log_analyzer

def ip_reputation(ip: str) -> dict:
    """Look up reputation and geolocation data for an IP address."""
    seed = int(hashlib.md5(ip.encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)

    known_bad = {
        "45.33.32.156": {
            "ip": "45.33.32.156", "country": "United States", "city": "Fremont, CA",
            "asn": "AS63949", "org": "Linode LLC", "risk_score": 72, "risk_level": "HIGH",
            "categories": ["scanning", "brute-force"], "last_seen_malicious": "2026-02-28",
            "malware_families": ["Mirai", "Hajime"], "abuse_reports": 147,
            "blocklists": ["Spamhaus DROP", "Emerging Threats"], "confidence": 0.89
        },
        "198.51.100.23": {
            "ip": "198.51.100.23", "country": "Russia", "city": "Moscow",
            "asn": "AS48666", "org": "Marosnet Telecom", "risk_score": 95, "risk_level": "CRITICAL",
            "categories": ["C2 server", "malware distribution", "phishing"],
            "last_seen_malicious": "2026-03-01",
            "malware_families": ["Cobalt Strike", "SystemBC"], "abuse_reports": 523,
            "blocklists": ["Spamhaus DROP", "Emerging Threats", "AlienVault OTX", "Feodo Tracker"],
            "confidence": 0.96
        },
    }
    if ip in known_bad:
        return known_bad[ip]

    countries = ["United States", "Germany", "Netherlands", "Singapore", "Japan", "Canada"]
    risk_score = rng.randint(5, 65)
    return {
        "ip": ip, "country": rng.choice(countries), "city": "Unknown",
        "asn": f"AS{rng.randint(1000, 99999)}",
        "org": rng.choice(["Amazon AWS", "DigitalOcean", "Google Cloud", "Cloudflare"]),
        "risk_score": risk_score,
        "risk_level": "LOW" if risk_score < 30 else "MEDIUM" if risk_score < 60 else "HIGH",
        "categories": [] if risk_score < 30 else [rng.choice(["scanning", "spam", "proxy"])],
        "malware_families": [], "abuse_reports": rng.randint(0, 50),
        "blocklists": [], "confidence": round(rng.uniform(0.6, 0.85), 2)
    }


def cve_lookup(cve_id: str) -> dict:
    """Look up a CVE by its ID. Returns description, CVSS, affected products, exploits."""
    known_cves = {
        "CVE-2024-3400": {
            "cve_id": "CVE-2024-3400",
            "description": "Command injection in GlobalProtect feature of Palo Alto PAN-OS. "
                           "Unauthenticated attacker can execute code with root privileges.",
            "cvss_score": 10.0, "cvss_severity": "CRITICAL",
            "affected_products": ["PAN-OS 11.1 < 11.1.2-h3", "PAN-OS 11.0 < 11.0.4-h1",
                                  "PAN-OS 10.2 < 10.2.9-h1"],
            "vendor": "Palo Alto Networks", "published_date": "2024-04-12",
            "exploit_available": True, "actively_exploited": True, "cisa_kev": True,
            "patch_available": True, "confidence": 0.99
        },
    }
    if cve_id in known_cves:
        return known_cves[cve_id]
    return {
        "cve_id": cve_id, "description": "CVE not found in database.",
        "cvss_score": None, "cvss_severity": "UNKNOWN", "affected_products": [],
        "exploit_available": False, "actively_exploited": False,
        "patch_available": False, "confidence": 0.0
    }


def log_analyzer(log_entries: str) -> dict:
    """Analyze raw log entries for security anomalies."""
    log_lines = [l.strip() for l in log_entries.strip().split("\n") if l.strip()]
    findings = []
    suspicious_ips = set()
    stats = {"total_lines": len(log_lines), "failed_logins": 0, "unique_ips": set()}

    suspicious_patterns = [
        (r"(?i)failed\s+(password|login|auth)", "BRUTE_FORCE", "HIGH", "Failed auth attempt"),
        (r"(?i)(\.\.[\\\\//]|path\s*traversal)", "PATH_TRAVERSAL", "CRITICAL", "Directory traversal attempt"),
        (r"(?i)(union\s+select|;\s*drop\s+|'\s*or\s+'1'\s*=\s*'1)", "SQL_INJECTION", "CRITICAL", "SQL injection attempt"),
        (r"(?i)(cmd\.exe|/bin/(sh|bash)|powershell)", "COMMAND_INJECTION", "CRITICAL", "Command injection attempt"),
        (r"(?i)(nmap|masscan|nikto|sqlmap)", "RECON", "MEDIUM", "Recon tool signature"),
    ]
    ip_pattern = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

    for i, line in enumerate(log_lines):
        found_ips = ip_pattern.findall(line)
        stats["unique_ips"].update(found_ips)
        if re.search(r"(?i)failed\s+(password|login|auth)", line):
            stats["failed_logins"] += 1
        for pattern, category, severity, description in suspicious_patterns:
            if re.search(pattern, line):
                findings.append({"line_number": i+1, "category": category,
                    "severity": severity, "description": description,
                    "source_ips": found_ips, "log_excerpt": line[:200]})
                suspicious_ips.update(found_ips)
                break

    if stats["failed_logins"] >= 5:
        findings.append({"line_number": "aggregate", "category": "BRUTE_FORCE_CAMPAIGN",
            "severity": "CRITICAL",
            "description": f"Brute force: {stats['failed_logins']} failed login attempts",
            "source_ips": list(suspicious_ips)})

    severity_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
    findings.sort(key=lambda f: severity_order.get(f["severity"], 4))

    return {
        "total_lines_analyzed": stats["total_lines"],
        "unique_ips_found": list(stats["unique_ips"]),
        "suspicious_ips": list(suspicious_ips),
        "summary": {"total_findings": len(findings),
            "critical": sum(1 for f in findings if f["severity"] == "CRITICAL"),
            "high": sum(1 for f in findings if f["severity"] == "HIGH")},
        "findings": findings[:10], "confidence": 0.85
    }

# Quick test
print("ip_reputation:", json.dumps(ip_reputation("45.33.32.156"), indent=2)[:200], "...")
print("cve_lookup:", cve_lookup("CVE-2024-3400")["cvss_severity"])
print("Tools 1-3 ready.")


In [ ]:
# Cell 4: Security Tools -- port_scanner, whois_lookup, threat_intel

def port_scanner(target: str) -> dict:
    """Simulated port scan with service detection."""
    seed = int(hashlib.md5(target.encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)

    known_targets = {
        "suspicious-domain.xyz": {
            "target": "suspicious-domain.xyz", "resolved_ip": "198.51.100.23",
            "scan_time": "3.78s", "host_status": "up",
            "open_ports": [
                {"port": 80, "service": "http", "version": "Apache 2.4.41"},
                {"port": 443, "service": "https", "version": "Apache 2.4.41"},
                {"port": 4444, "service": "unknown", "version": "unknown"},
                {"port": 8443, "service": "https-alt", "version": "Cobalt Strike Beacon"},
            ],
            "os_guess": "Linux 4.x (Debian)", "confidence": 0.88
        },
    }
    if target in known_targets:
        return known_targets[target]

    common_ports = [(22, "ssh", "OpenSSH 8.2p1"), (80, "http", "nginx 1.18"),
                    (443, "https", "nginx 1.18"), (8080, "http-proxy", "Tomcat 9.0")]
    selected = rng.sample(common_ports, k=rng.randint(2, 4))
    return {
        "target": target, "scan_time": f"{rng.uniform(1.0, 5.0):.2f}s", "host_status": "up",
        "open_ports": [{"port": p, "service": s, "version": v} for p, s, v in sorted(selected)],
        "os_guess": rng.choice(["Linux 5.x", "Windows Server 2022"]),
        "confidence": round(rng.uniform(0.70, 0.90), 2)
    }


def whois_lookup(domain: str) -> dict:
    """Look up WHOIS registration data for a domain."""
    known_domains = {
        "suspicious-domain.xyz": {
            "domain": "suspicious-domain.xyz", "registrar": "NameSilo LLC",
            "creation_date": "2026-01-15", "expiration_date": "2027-01-15",
            "domain_age_days": 46,
            "registrant": {"name": "REDACTED FOR PRIVACY", "country": "IS"},
            "nameservers": ["ns1.shady-dns-provider.ru", "ns2.shady-dns-provider.ru"],
            "dnssec": "unsigned", "resolved_ips": ["198.51.100.23"],
            "red_flags": ["Domain only 46 days old", "Privacy-protected registration",
                          "Nameservers in different country", ".xyz TLD commonly abused"],
            "confidence": 0.92
        },
    }
    if domain in known_domains:
        return known_domains[domain]

    seed = int(hashlib.md5(domain.encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)
    return {
        "domain": domain,
        "registrar": rng.choice(["GoDaddy", "Namecheap", "Cloudflare"]),
        "creation_date": "2023-06-15", "domain_age_days": 990,
        "registrant": {"name": "REDACTED", "country": "US"},
        "nameservers": ["ns1.example.com", "ns2.example.com"],
        "red_flags": [], "confidence": 0.70
    }


def threat_intel(ioc: str) -> dict:
    """Enrich an IOC (IP, hash, or domain) with threat intelligence."""
    if re.match(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$", ioc):
        ioc_type = "ip"
    elif re.match(r"^[a-fA-F0-9]{32}$", ioc) or re.match(r"^[a-fA-F0-9]{64}$", ioc):
        ioc_type = "hash"
    else:
        ioc_type = "domain"

    known_iocs = {
        "198.51.100.23": {
            "ioc": "198.51.100.23", "ioc_type": "ip", "threat_score": 95,
            "tags": ["C2", "cobalt-strike", "ransomware-infrastructure"],
            "associated_campaigns": ["BlackCat-affiliate"],
            "associated_malware": ["Cobalt Strike", "SystemBC", "BlackCat"],
            "attack_patterns": ["T1071 - Application Layer Protocol", "T1486 - Data Encrypted for Impact"],
            "attribution": "Suspected Eastern European ransomware affiliate group",
            "recommendations": ["IMMEDIATE: Block at all perimeter controls",
                                "Hunt for Cobalt Strike beacons on endpoints",
                                "Engage incident response if connections found"],
            "sources": ["CISA", "Mandiant", "CrowdStrike"], "confidence": 0.96
        },
        "suspicious-domain.xyz": {
            "ioc": "suspicious-domain.xyz", "ioc_type": "domain", "threat_score": 91,
            "tags": ["C2", "phishing", "newly-registered"],
            "associated_campaigns": ["BlackCat-affiliate"],
            "associated_malware": ["Cobalt Strike", "SystemBC"],
            "attack_patterns": ["T1566 - Phishing", "T1071 - Application Layer Protocol"],
            "attribution": "Suspected Eastern European ransomware affiliate group",
            "recommendations": ["Block domain at DNS and proxy",
                                "Search email logs for this domain",
                                "If visited, initiate incident response"],
            "sources": ["AlienVault OTX", "URLhaus"], "confidence": 0.93
        },
    }
    if ioc in known_iocs:
        return known_iocs[ioc]

    seed = int(hashlib.md5(ioc.encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)
    score = rng.randint(10, 50)
    return {
        "ioc": ioc, "ioc_type": ioc_type, "threat_score": score,
        "tags": [] if score < 30 else ["suspicious"],
        "associated_campaigns": [], "associated_malware": [],
        "recommendations": ["Continue monitoring"], "confidence": round(rng.uniform(0.5, 0.75), 2)
    }

# Quick test
print("port_scanner:", port_scanner("suspicious-domain.xyz")["open_ports"][3])
print("whois:", whois_lookup("suspicious-domain.xyz")["red_flags"][:2])
print("threat_intel:", threat_intel("198.51.100.23")["threat_score"])
print("Tools 4-6 ready.")


---
## Part 2: The Tool Registry
---

The **tool registry** maps tool names to functions and describes each tool for the LLM.
This is the same concept as OpenAI function calling schemas or Anthropic tool use definitions.

In [ ]:
# Cell 5: Tool Registry
TOOL_REGISTRY = {
    "ip_reputation": {
        "function": ip_reputation,
        "description": "Look up reputation and geolocation for an IP address. "
                       "Returns risk score, country, ASN, malware associations.",
        "parameters": {"ip": {"type": "string", "description": "IP address to look up"}},
        "returns": "JSON with risk_score, risk_level, malware_families, blocklists."
    },
    "cve_lookup": {
        "function": cve_lookup,
        "description": "Look up a CVE by ID. Returns CVSS score, affected products, exploit status.",
        "parameters": {"cve_id": {"type": "string", "description": "CVE identifier (e.g. 'CVE-2024-3400')"}},
        "returns": "JSON with cvss_score, cvss_severity, affected_products, exploit_available."
    },
    "log_analyzer": {
        "function": log_analyzer,
        "description": "Analyze raw log entries for security anomalies. Detects brute force, SQLi, XSS, etc.",
        "parameters": {"log_entries": {"type": "string", "description": "Raw log entries, one per line."}},
        "returns": "JSON with suspicious_ips, findings list with severity levels."
    },
    "port_scanner": {
        "function": port_scanner,
        "description": "Scan a target for open ports with service and version detection.",
        "parameters": {"target": {"type": "string", "description": "IP or domain to scan."}},
        "returns": "JSON with open_ports list, os_guess, host_status."
    },
    "whois_lookup": {
        "function": whois_lookup,
        "description": "Look up WHOIS registration data for a domain.",
        "parameters": {"domain": {"type": "string", "description": "Domain name to look up."}},
        "returns": "JSON with registrar, creation_date, registrant, red_flags."
    },
    "threat_intel": {
        "function": threat_intel,
        "description": "Enrich an IOC (IP, hash, or domain) with threat intelligence.",
        "parameters": {"ioc": {"type": "string", "description": "IP, file hash, or domain."}},
        "returns": "JSON with threat_score, tags, campaigns, attribution, recommendations."
    },
}

def get_tool_descriptions() -> str:
    """Format tool descriptions for the system prompt."""
    lines = []
    for name, info in TOOL_REGISTRY.items():
        params_str = ", ".join(
            f'"{k}": <{v["type"]}> -- {v["description"]}' for k, v in info["parameters"].items())
        lines.append(f"Tool: {name}\n  Description: {info['description']}\n"
                     f"  Parameters: {{{params_str}}}\n  Returns: {info['returns']}\n")
    return "\n".join(lines)

def execute_tool(tool_name: str, args: dict) -> dict:
    """Execute a tool by name with given arguments."""
    if tool_name not in TOOL_REGISTRY:
        return {"error": f"Unknown tool: {tool_name}", "available_tools": list(TOOL_REGISTRY.keys())}
    try:
        return TOOL_REGISTRY[tool_name]["function"](**args)
    except Exception as e:
        return {"error": f"Tool execution failed: {str(e)}", "tool": tool_name}

print(f"Tool registry ready: {list(TOOL_REGISTRY.keys())}")


---
## Part 3: Building the Agent Loop
---

Now we build the core agent. **The challenge:** Small models (3B params) are less
reliable at structured output than GPT-4 or Claude. We handle this with:

1. **XML-based tool calling format** -- easier for small models to parse
2. **Few-shot examples** in the system prompt
3. **Robust parsing** with regex fallbacks
4. **Maximum iterations** to prevent infinite loops

### Tool Call Format

```
<tool_call>{"tool": "tool_name", "args": {"param": "value"}}</tool_call>
```
When finished: `<final_answer>The analysis goes here.</final_answer>`

In [ ]:
# Cell 6: System Prompt and Generation Helper
SYSTEM_PROMPT = f"""You are a Security Investigation Agent. You help security analysts investigate threats by using specialized security tools.

## Available Tools

{get_tool_descriptions()}

## How to Use Tools

When you need information, call a tool using this EXACT format:

<tool_call>{{"tool": "tool_name", "args": {{"param_name": "value"}}}}</tool_call>

IMPORTANT RULES:
1. Call ONE tool at a time.
2. After calling a tool, STOP and wait for the result.
3. When you have enough information, provide your final analysis using:

<final_answer>
Your complete analysis here.
</final_answer>

## Example

User: What can you tell me about IP 8.8.8.8?
Assistant: I will look up the reputation data for this IP.

<tool_call>{{"tool": "ip_reputation", "args": {{"ip": "8.8.8.8"}}}}</tool_call>

[After receiving result]
<final_answer>
IP 8.8.8.8 is Google's public DNS resolver. Risk score is very low.
</final_answer>

## Investigation Approach
1. Start with the most relevant tool for the query
2. Use results to guide follow-up investigations
3. Cross-reference findings across multiple tools
4. Synthesize into a clear, actionable report

Always explain your reasoning before each tool call."""


def generate_response(messages, max_new_tokens=512):
    """Generate a response from the model."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.3, top_p=0.9,
            do_sample=True, repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(f"System prompt: {len(SYSTEM_PROMPT)} chars. Generation helper ready.")


In [ ]:
# Cell 7: Tool Call Parser
def parse_tool_call(response: str) -> dict | None:
    """Parse a tool call from model output. Handles common failure modes."""
    # Strategy 1: Proper <tool_call> tags
    match = re.search(r"<tool_call>(.*?)</tool_call>", response, re.DOTALL)
    if match:
        raw = match.group(1).strip()
        try:
            parsed = json.loads(raw)
            if "tool" in parsed and "args" in parsed:
                return parsed
        except json.JSONDecodeError:
            pass
        try:
            fixed = raw.replace("'", '"')
            parsed = json.loads(fixed)
            if "tool" in parsed and "args" in parsed:
                return parsed
        except json.JSONDecodeError:
            pass

    # Strategy 2: JSON-like pattern anywhere in response
    match = re.search(r'\{\s*"tool"\s*:\s*"(\w+)"\s*,\s*"args"\s*:\s*(\{[^}]+\})\s*\}',
                      response, re.DOTALL)
    if match:
        try:
            args = json.loads(match.group(2))
            return {"tool": match.group(1), "args": args}
        except json.JSONDecodeError:
            pass

    # Strategy 3: Tool name + quoted argument (very lenient)
    for tool_name in TOOL_REGISTRY:
        if tool_name in response:
            params = TOOL_REGISTRY[tool_name]["parameters"]
            param_name = list(params.keys())[0]
            arg_match = re.search(tool_name + r'.*?["\'](.*?)["\'\n]', response, re.DOTALL)
            if arg_match:
                return {"tool": tool_name, "args": {param_name: arg_match.group(1)}}

    return None


def parse_final_answer(response: str) -> str | None:
    """Extract the final answer from model output."""
    match = re.search(r"<final_answer>(.*?)</final_answer>", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    if not parse_tool_call(response) and len(response) > 100:
        return response.strip()
    return None

# Quick test
test = '<tool_call>{"tool": "ip_reputation", "args": {"ip": "1.2.3.4"}}</tool_call>'
print(f"Parse test: {parse_tool_call(test)}")
print("Parser ready.")


In [ ]:
# Cell 8: The Agent Loop
def run_agent(user_query: str, max_iterations: int = 6, verbose: bool = True) -> str:
    """Run the security agent on a user query."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    if verbose:
        print("=" * 60)
        print(f"SECURITY AGENT -- Query: {user_query[:80]}")
        print("=" * 60)

    tools_called = []

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {iteration + 1}/{max_iterations} ---")

        response = generate_response(messages, max_new_tokens=512)
        if verbose:
            print(f"Agent:\n{response}")

        final = parse_final_answer(response)
        tool_call = parse_tool_call(response)

        if final and not tool_call:
            if verbose:
                print(f"\nINVESTIGATION COMPLETE. Tools used: {tools_called}")
            return final

        if tool_call:
            tool_name = tool_call["tool"]
            tool_args = tool_call["args"]
            if verbose:
                print(f"\n>> TOOL CALL: {tool_name}({json.dumps(tool_args)})")

            result = execute_tool(tool_name, tool_args)
            tools_called.append(tool_name)

            if verbose:
                preview = json.dumps(result, indent=2, default=str)
                print(f">> RESULT:\n{preview[:500]}")

            messages.append({"role": "assistant", "content": response})
            messages.append({
                "role": "user",
                "content": f"Tool result for {tool_name}:\n{json.dumps(result, indent=2, default=str)}\n\n"
                           f"Continue investigating or provide your <final_answer>."
            })
            continue

        # No tool call or final answer -- nudge the model
        if verbose:
            print("[No tool call or final answer. Nudging...]")
        messages.append({"role": "assistant", "content": response})
        messages.append({
            "role": "user",
            "content": "Use <tool_call>...</tool_call> or <final_answer>...</final_answer>."
        })

    # Max iterations -- force summary
    messages.append({"role": "user", "content": "Provide your <final_answer> now."})
    response = generate_response(messages, max_new_tokens=512)
    final = parse_final_answer(response)
    if verbose:
        print(f"\nINVESTIGATION COMPLETE (max iterations). Tools used: {tools_called}")
    return final if final else response

print("Agent loop ready. Use: run_agent('your security question here')")


---
## Part 4: Demo Investigations
---

Let us put the agent to work with two investigations to see how it reasons and chains tools.

In [ ]:
# Demo 1: Simple IP Investigation
print("\n" + "#" * 60)
print("# DEMO 1: Simple IP Investigation")
print("#" * 60)

demo1_result = run_agent(
    "What can you tell me about IP 45.33.32.156? Is it malicious?",
    max_iterations=4, verbose=True
)

print("\n" + "=" * 60)
print("FINAL REPORT:")
print("=" * 60)
print(demo1_result)


### Demo 1 Discussion

The agent should call `ip_reputation` first, possibly followed by `threat_intel`.
Did it explain its reasoning? Did it correctly identify the IP as high-risk?
If formatting failed, the retry/nudge logic should have recovered.

In [ ]:
# Demo 2: Incident Triage (Log Analysis + IP Investigation)
print("\n" + "#" * 60)
print("# DEMO 2: Incident Triage")
print("#" * 60)

suspicious_logs = """2026-03-01 08:14:22 sshd[12001]: Failed password for root from 198.51.100.23 port 4822 ssh2
2026-03-01 08:14:23 sshd[12002]: Failed password for admin from 198.51.100.23 port 4823 ssh2
2026-03-01 08:14:24 sshd[12003]: Failed password for root from 198.51.100.23 port 4824 ssh2
2026-03-01 08:14:25 sshd[12004]: Failed password for root from 198.51.100.23 port 4825 ssh2
2026-03-01 08:14:26 sshd[12005]: Failed password for admin from 198.51.100.23 port 4826 ssh2
2026-03-01 08:14:27 sshd[12006]: Failed password for root from 198.51.100.23 port 4827 ssh2
2026-03-01 08:15:01 httpd[5501]: 203.0.113.50 - - "GET /admin/../../../etc/passwd HTTP/1.1" 403
2026-03-01 08:15:05 httpd[5502]: 203.0.113.50 - - "POST /login HTTP/1.1" 200 -- 'admin' OR '1'='1'
2026-03-01 08:15:10 httpd[5503]: 203.0.113.50 - - "GET /admin/cmd.exe?/c+whoami HTTP/1.1" 404
2026-03-01 08:16:00 kernel: [UFW BLOCK] SRC=198.51.100.23 DST=10.0.1.5 PROTO=TCP DPT=4444"""

demo2_result = run_agent(
    f"Analyze these logs from our production server and identify potential threats. "
    f"Investigate any suspicious IPs you find.\n\nLogs:\n{suspicious_logs}",
    max_iterations=6, verbose=True
)

print("\n" + "=" * 60)
print("FINAL REPORT:")
print("=" * 60)
print(demo2_result)


### Demo 2 Discussion

The agent should analyze logs first, then investigate suspicious IPs (198.51.100.23, 203.0.113.50).
This demonstrates **chaining**: output from log analysis feeds into IP reputation lookups.
Watch for: brute force detection, web attack identification, and the blocked C2 connection on port 4444.

---
## Part 5: Agent Improvements
---

In [ ]:
# Cell 9: Improved Agent with Memory and Error Handling
class SecurityAgent:
    """Security agent with conversation memory and error handling."""

    def __init__(self, model, tokenizer, tool_registry, verbose=True):
        self.model = model
        self.tokenizer = tokenizer
        self.tools = tool_registry
        self.verbose = verbose
        self.memory = []  # Past investigation summaries

    def _generate(self, messages, max_new_tokens=512):
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=0.3, top_p=0.9,
                do_sample=True, repetition_penalty=1.1, pad_token_id=self.tokenizer.eos_token_id)
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    def _execute_tool(self, tool_name, args):
        if tool_name not in self.tools:
            return {"error": f"Unknown tool '{tool_name}'", "available_tools": list(self.tools.keys())}
        try:
            return self.tools[tool_name]["function"](**args)
        except Exception as e:
            return {"error": f"Tool failed: {str(e)}"}

    def investigate(self, query, max_iterations=6):
        """Run a full investigation with memory."""
        tools_used = []

        # Build memory context from past investigations
        memory_ctx = ""
        if self.memory:
            memory_ctx = "\n\nPrevious investigations:\n"
            for past in self.memory[-3:]:
                memory_ctx += f"- {past['query'][:80]}: {past['summary'][:100]}\n"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT + memory_ctx},
            {"role": "user", "content": query}
        ]

        if self.verbose:
            print(f"{'=' * 60}\nSECURITY AGENT v2 -- {query[:60]}\n{'=' * 60}")

        for iteration in range(max_iterations):
            if self.verbose:
                print(f"\n--- Iteration {iteration + 1}/{max_iterations} ---")
            response = self._generate(messages)
            if self.verbose:
                print(f"Agent:\n{response}")

            final = parse_final_answer(response)
            tool_call = parse_tool_call(response)

            if final and not tool_call:
                self.memory.append({"query": query, "summary": final[:200], "tools": tools_used})
                if self.verbose:
                    print(f"\nCOMPLETE. Tools: {tools_used}")
                return final

            if tool_call:
                t_name, t_args = tool_call["tool"], tool_call["args"]
                if self.verbose:
                    print(f">> TOOL: {t_name}({json.dumps(t_args)})")
                result = self._execute_tool(t_name, t_args)
                tools_used.append(t_name)
                if self.verbose:
                    print(f">> RESULT:\n{json.dumps(result, indent=2, default=str)[:400]}")
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user",
                    "content": f"Tool result for {t_name}:\n{json.dumps(result, indent=2, default=str)}\n\n"
                               f"Continue or provide <final_answer>."})
                continue

            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user",
                "content": "Use <tool_call>...</tool_call> or <final_answer>...</final_answer>."})

        messages.append({"role": "user", "content": "Provide your <final_answer> now."})
        response = self._generate(messages)
        final = parse_final_answer(response) or response
        self.memory.append({"query": query, "summary": final[:200], "tools": tools_used})
        return final

agent = SecurityAgent(model, tokenizer, TOOL_REGISTRY, verbose=True)
print("SecurityAgent v2 ready. Use: agent.investigate('your query')")


In [ ]:
# Cell 10: Test the Improved Agent
print("#" * 60)
print("# Testing Agent v2 with Memory")
print("#" * 60)

result = agent.investigate("Check the reputation of IP 198.51.100.23", max_iterations=3)
print("\nFINAL REPORT:")
print(result)

print(f"\nAgent memory: {len(agent.memory)} investigation(s) stored.")


---
## Part 6: Excessive Agency -- The Critical Risk
---

Our agent can **read** data: look up IPs, search CVEs, analyze logs. These are
**safe, read-only operations**. But what if we gave it tools that **take action**?

### Dangerous Tools (Imagine These Existed)

```python
def block_ip(ip, firewall="perimeter"):
    """Block an IP at the firewall. IRREVERSIBLE without manual review."""

def quarantine_host(hostname):
    """Isolate a host from the network. Disrupts user productivity."""

def disable_user(username):
    """Disable an Active Directory account. User cannot work."""
```

### OWASP LLM08: Excessive Agency

> *"Excessive Agency is the vulnerability that enables damaging actions to be
> performed in response to unexpected, ambiguous, or manipulated outputs from
> an LLM, regardless of what is causing the LLM to malfunction."*

### Mitigation: Human-in-the-Loop

For any tool that **changes state**, the agent should:
1. **Recommend** the action ("I recommend blocking 198.51.100.23")
2. **Explain** why ("Risk score 95/100, associated with ransomware")
3. **Wait for human approval** before executing
4. **Log the decision** (who approved, when, why)

**Think about it:** Would you trust a 3B parameter model to decide whether to
block an IP on your production firewall? What about a model that has been
prompt-injected? This is why **excessive agency** is in the OWASP Top 10.

In [ ]:
# Cell 11: Human-in-the-Loop Demonstration
def block_ip_with_approval(ip: str, reason: str, auto_approve: bool = False) -> dict:
    """Simulates a firewall block with human-in-the-loop approval."""
    print("\n" + "!" * 60)
    print("!! HUMAN APPROVAL REQUIRED !!")
    print("!" * 60)
    print(f"  Action: BLOCK IP at perimeter firewall")
    print(f"  Target: {ip}")
    print(f"  Reason: {reason}")
    print(f"  Impact: All traffic from {ip} will be dropped")

    if auto_approve:
        print("  [AUTO-APPROVED for demonstration]")
        approved = True
    else:
        print("  In production: Analyst would approve/reject via SOC dashboard")
        approved = True  # Simulated

    print("!" * 60)

    if approved:
        return {"action": "block_ip", "status": "EXECUTED", "target": ip,
                "approved_by": "analyst@soc.example.com",
                "rule_id": f"FW-{random.randint(10000, 99999)}",
                "note": "SIMULATED block for demonstration."}
    return {"action": "block_ip", "status": "REJECTED", "target": ip}

# Demonstrate
print("Demonstrating human-in-the-loop for a 'write' action:\n")
result = block_ip_with_approval(
    ip="198.51.100.23",
    reason="CRITICAL risk (95/100). C2 server, BlackCat ransomware. Active internal connections.",
    auto_approve=True
)
print("\nResult:", json.dumps(result, indent=2))


---
## Part 7: Student Exercises
---

Now it is your turn. Complete the following exercises to deepen your understanding
of agentic AI and security tool calling.

### Exercise 1: Add a New Security Tool

Create a `hash_lookup` tool, add it to `TOOL_REGISTRY`, and test with the agent.

In [ ]:
# Exercise 1: Add hash_lookup tool
# YOUR CODE HERE
# def hash_lookup(file_hash: str) -> dict:
#     """Look up a file hash in threat intel databases."""
#     # Create known malicious hashes + fallback for unknown
#     pass
#
# Add to TOOL_REGISTRY, then test:
# agent.investigate("Check if this hash is malicious: a1b2c3d4e5f6...")

print("Exercise 1: Implement hash_lookup and add to TOOL_REGISTRY")


### Exercise 2: Multi-Tool Investigation Scenario

Create a scenario requiring **at least 3 tool calls**. Run it with the agent.

In [ ]:
# Exercise 2: Multi-tool scenario
# YOUR CODE HERE
# scenario = "Write a realistic scenario requiring whois + ip_reputation + threat_intel..."
# result = agent.investigate(scenario, max_iterations=6)

print("Exercise 2: Create a scenario requiring 3+ tool calls.")


### Exercise 3: Add a Confirmation Step for Write Actions

Modify `SecurityAgent._execute_tool` to check for `requires_approval: True`
in the registry and pause for confirmation before executing.

In [ ]:
# Exercise 3: Confirmation step for write actions
# YOUR CODE HERE
# Hint: In _execute_tool, check self.tools[tool_name].get("requires_approval", False)
# Then add block_ip_with_approval to TOOL_REGISTRY with requires_approval: True

print("Exercise 3: Add approval gates for dangerous tools.")


---
## Key Takeaways
---

### What You Built
- A **functional AI security agent** with tool calling, parsing, and an agentic loop
- Six security tools with realistic simulated data
- Memory, error handling, and human-in-the-loop safety gates

### Key Lessons

1. **The agentic loop is simple.** Generate -> parse -> execute -> feed back.
   The complexity is in making it reliable.

2. **Small models can do tool calling,** but need clear formats (XML tags),
   few-shot examples, and robust parsing with fallbacks.

3. **Excessive agency is dangerous.** Read-only tools are safe to automate.
   Write/action tools need human approval gates.

---

*Lab 9 Complete. Derek Banks and Dr. Brian Fehrman, 2026.*